# Classification of Wikipedia Articles
Wikipedia is an encyclopedia that covers a large amount of diverse topics. All articles are created, corrected and updated by individuals. The goal is to correctly document as many topics as possible by collecting the knowledge of a large number of people. However, some articles stand out due to their completeness, scope and presentation, and for this they are marked with the distinction of the Excellent Article. 

As part of the Natural Language Processing lecture, a classification of Wikipedia articles is to be carried out as a sub-task of an assignment with the goal of being able to identify excellent articles. This notebook contains the code to accomplish this goal and is structured as follows:

1. [Imports](#1-imports)
2. [Load Preprocessed Data](#2-load-preprocessed-data)
3. [Prepare Dataset for Neural Network](#3-prepare-data-for-neural-network) <br>
	3.1 [Tokenize Words](#31-tokenize-words) <br>
	3.2 [Clip Text Length](#32-clip-text-length) <br>
4. [Split Dataset](#4-split-dataset)
5. [Train Neural Network](#5-train-neural-network) <br>
	5.1 [Define Neural Network Model]() <br>
	5.2 [Compile Neural Network Model]() <br>
	5.3 [Train the Neural Network]() <br>
6. [Validation of Results]()
7. [Conclusion]()


## 1. Imports
Import the requiered libraties into the notebook.
If some libraries are not installed, you can use the `requierements.txt` and run
```
$ pip install -r requirements.txt
```
in the terminal.

In [3]:
# Import data science library
import pandas as pd
import numpy as np

# Import Pre-Processing libraries
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Import neural network framework & layers
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Conv1D, LSTM, Dense
from tensorflow.keras.layers import BatchNormalization, Dropout, MaxPooling1D
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.regularizers import L1L2

# Import classification metrics
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, accuracy_score
from imblearn.metrics import geometric_mean_score

# Import visualization libraries
import plotly.graph_objects as go
from prettytable import PrettyTable

## 2. Load Preprocessed Data

In [6]:
dataframe = pd.read_pickle("../../Data/processed_dataset.pkl")

In [7]:
X = np.array(dataframe["text"].values)
y = np.asanyarray(dataframe["label"].values).astype(np.int16)

In [8]:
np.unique(y, return_counts=True)

(array([0, 1], dtype=int16), array([4193, 2794]))

## 3. Prepare Data for Neural Network

### 3.1 Tokenize Words

In [9]:
tokenizer = Tokenizer(
    num_words=10000,
    filters='!"„“#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'
)
tokenizer.fit_on_texts(X)

### 3.2 Clip Text Length

In [10]:
X_token = pad_sequences(tokenizer.texts_to_sequences(X), maxlen=10000)

## 4. Split Dataset

In [11]:
X_train, X_rest, y_train, y_rest = train_test_split(
    X_token, 
    y,
    stratify=y, 
    test_size=0.3,
    random_state=456
)

X_test, X_val, y_test, y_val = train_test_split(
    X_rest,
    y_rest,
    stratify=y_rest,
    test_size=0.5,
    random_state=456
)

In [12]:
print(len(X_train), len(X_test), len(X_val))

4890 1048 1049


## 5. Train Neural Network

### 5.1 Define Neural Network Model

In [13]:
model = tf.keras.Sequential([
    Embedding(input_dim=10000, output_dim=256, embeddings_regularizer=L1L2(0, 0.001)), # Add regularization
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=4),
    BatchNormalization(),
    Conv1D(filters=64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=4),
    BatchNormalization(),
    Dropout(0.5),
    LSTM(64, return_sequences=True, kernel_regularizer=L1L2(0, 0.001)),
    Dropout(0.5),
    LSTM(32, kernel_regularizer=L1L2(0, 0.001)),
    Dropout(0.5),
    Dense(32),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

opt = tf.keras.optimizers.legacy.Adam(learning_rate=0.0005)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 256)         2560000   
                                                                 
 conv1d (Conv1D)             (None, None, 128)         163968    
                                                                 
 max_pooling1d (MaxPooling1D  (None, None, 128)        0         
 )                                                               
                                                                 
 batch_normalization (BatchN  (None, None, 128)        512       
 ormalization)                                                   
                                                                 
 conv1d_1 (Conv1D)           (None, None, 64)          41024     
                                                                 
 max_pooling1d_1 (MaxPooling  (None, None, 64)         0

### 5.2 Compile Neural Network Model

In [55]:
model.compile(
    loss='binary_crossentropy', 
    optimizer=opt, 
    metrics=[
        'binary_accuracy'
    ]
)

### 5.3 Train Neural Network

In [56]:
earlystopper = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=0.000001, verbose=1, cooldown=5)

history = model.fit(
    X_train, 
    y_train,
    validation_data=(X_val, y_val),
    epochs=300, 
    batch_size=100,
    verbose=1,
    shuffle=True,
    callbacks=[earlystopper, reduce_lr]
)

Epoch 1/300
49/49 [==============================] - 494s 10s/step - loss: 2.3665 - binary_accuracy: 0.6918 - val_loss: 2.3358 - val_binary_accuracy: 0.5996 - lr: 5.0000e-04
Epoch 2/300
49/49 [==============================] - 479s 10s/step - loss: 1.4528 - binary_accuracy: 0.9454 - val_loss: 2.9756 - val_binary_accuracy: 0.5996 - lr: 5.0000e-04
Epoch 3/300
49/49 [==============================] - 482s 10s/step - loss: 1.0655 - binary_accuracy: 0.9763 - val_loss: 2.8999 - val_binary_accuracy: 0.5996 - lr: 5.0000e-04
Epoch 4/300
49/49 [==============================] - ETA: 0s - loss: 0.8238 - binary_accuracy: 0.9867
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
49/49 [==============================] - 485s 10s/step - loss: 0.8238 - binary_accuracy: 0.9867 - val_loss: 2.5789 - val_binary_accuracy: 0.5996 - lr: 5.0000e-04
Epoch 5/300
49/49 [==============================] - 490s 10s/step - loss: 0.6877 - binary_accuracy: 0.9912 - val_loss: 1.6804 - val_binar

In [57]:
fig = go.Figure(
    data = [
        go.Scatter(y=history.history['loss'], name="train"),
        go.Scatter(y=history.history['val_loss'], name="val"),
    ],
    layout = {"yaxis": {"title": "Loss [BCE]"}, "xaxis": {"title": "Epoch"}, "title": "Model Loss over Epochs"}
)
fig.show()

In [14]:
# model.save('./99_Saved Models/03_neural_network_model.h5')
model = tf.keras.models.load_model('./99_Saved Models/03_neural_network_model.h5')

## 6. Validation of Results

In [15]:
y_test_predictions = (np.array(model.predict(X_test)) >= 0.5).astype(int)
f1score = f1_score(y_test, y_test_predictions)
gm = geometric_mean_score(y_test, y_test_predictions, average="binary")
auc = roc_auc_score(y_test, y_test_predictions, average="weighted")
precision = precision_score(y_test, y_test_predictions)
recall = recall_score(y_test, y_test_predictions)
acc = accuracy = accuracy_score(y_test,  y_test_predictions)

2023-07-19 15:27:53.751349: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


33/33 [==============================] - 3s 60ms/step


In [16]:
data = [["F1-Score", "G-Mean", "AUC", "Precision", "Recall", "Accuracy"], [f1score, gm, auc, precision, recall, acc]] # Create list with values
table = PrettyTable(data[0]) # Generate table with metrics
table.add_rows(data[1:]) # Add data to table
print(table) # Show table

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|      F1-Score      |       G-Mean       |        AUC         |     Precision      |       Recall       |      Accuracy      |
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
| 0.9751479289940829 | 0.9805140738180307 | 0.9805180022083012 | 0.9671361502347418 | 0.9832935560859188 | 0.9799618320610687 |
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
